# STEP 2 — Calorie Estimation (Revisi: Unit Conversion)

Perbaikan dari versi sebelumnya:
- **Versi lama:** kalori semua bahan dijumlah mentah → hasil tidak realistis (2.000-3.000 kkal/resep)
- **Versi baru:** setiap bahan dikonversi ke gram terlebih dahulu, kalori dihitung per 100g, lalu dibagi estimasi jumlah porsi

**Pipeline:**
1. Ekstrak nama bahan + satuan + kuantitas
2. Konversi satuan → gram menggunakan tabel konversi
3. Hitung kalori = (gram / 100) × kalori_per_100g
4. Bagi dengan estimasi jumlah porsi per resep
5. Filter resep dengan estimasi kalori realistis

**Input:** `data/resep_clean.csv`, `data/nutrition.csv`

**Output:** `data/dataset_final_resep_nutrisi.csv`

In [1]:
import pandas as pd
import numpy as np
import re
from rapidfuzz import process, fuzz
from tqdm import tqdm

tqdm.pandas()
print("Library siap.")

Library siap.


In [2]:
# ============================================================
# LOAD DATA
# ============================================================

df_resep   = pd.read_csv("../data/resep_clean.csv")
df_nutrisi = pd.read_csv("../data/nutrition.csv")

print(f"Resep   : {len(df_resep):,}")
print(f"Nutrisi : {len(df_nutrisi):,}")

Resep   : 11,514
Nutrisi : 1,346


In [3]:
# ============================================================
# TABEL KONVERSI SATUAN → GRAM
# Sumber: standar ukuran rumah tangga Indonesia (URT)
# Semua nilai dalam gram
# ============================================================

KONVERSI_SATUAN = {
    # Takaran sendok
    "sdm":        15.0,    # sendok makan
    "sdt":         5.0,    # sendok teh
    "sendok makan":15.0,
    "sendok teh":   5.0,

    # Takaran gelas/cup
    "gelas":      200.0,
    "cup":        240.0,
    "ml":          1.0,
    "cc":          1.0,
    "liter":    1000.0,
    "l":        1000.0,

    # Takaran berat
    "gram":        1.0,
    "gr":          1.0,
    "g":           1.0,
    "kg":       1000.0,
    "ons":       100.0,
    "mg":          0.001,

    # Satuan cacahan — estimasi berat per unit (URT)
    "butir":      55.0,    # telur ukuran sedang
    "buah":       80.0,    # buah/sayur ukuran sedang
    "biji":       10.0,    # biji kecil
    "bh":         80.0,
    "btr":        55.0,
    "siung":       5.0,    # bawang putih/merah per siung
    "lembar":     10.0,    # daun-daunan
    "lbr":        10.0,
    "ikat":      100.0,    # sayuran seikat
    "batang":     15.0,    # serai, daun bawang
    "tangkai":    10.0,
    "ruas":        5.0,    # jahe, kunyit, lengkuas per ruas
    "jari":        5.0,
    "ekor":      800.0,    # ayam/ikan utuh
    "potong":     80.0,    # potongan daging/ayam
    "ptg":        80.0,
    "iris":       20.0,    # irisan tipis
    "helai":      10.0,
    "lonjor":    200.0,    # tahu/tempe per lonjor
    "papan":     200.0,    # tempe per papan
    "keping":     30.0,
    "bungkus":   200.0,    # bungkus kecil
    "sachet":     10.0,
    "kantong":   200.0,
    "genggam":    30.0,
    "lusin":     660.0,    # 12 butir telur
    "ons":       100.0,
}

# Default jika satuan tidak dikenali (50g = konservatif)
DEFAULT_GRAM = 50.0

# Estimasi porsi per resep berdasarkan konteks
# (resep Indonesia umumnya untuk 2-6 orang)
DEFAULT_PORSI = 4.0

print(f"Tabel konversi: {len(KONVERSI_SATUAN)} satuan terdaftar")
print(f"Default gram  : {DEFAULT_GRAM}g (untuk satuan tidak dikenal)")
print(f"Default porsi : {DEFAULT_PORSI} porsi per resep")

Tabel konversi: 42 satuan terdaftar
Default gram  : 50.0g (untuk satuan tidak dikenal)
Default porsi : 4.0 porsi per resep


In [4]:
# ============================================================
# FUNGSI EKSTRAK KUANTITAS, SATUAN, DAN NAMA BAHAN
# ============================================================

# Pola angka: 1, 2, 1/2, 1.5, dll
POLA_ANGKA = r"(\d+[\.,]?\d*(?:[/\\]\d+)?)"

# Satuan yang dikenal (untuk regex)
SATUAN_REGEX = "|".join(sorted(KONVERSI_SATUAN.keys(), key=len, reverse=True))

STOPWORDS_BAHAN = {
    "haluskan", "cincang", "potong", "geprek", "memarkan",
    "rebus", "goreng", "tumis", "bakar", "kukus", "sangrai",
    "kupas", "cuci", "bersih", "segar", "kering", "matang",
    "mentah", "tipis", "kasar", "halus", "besar", "kecil",
    "sedang", "panjang", "pendek", "bulat", "sesuai",
    "secukupnya", "sck", "optional", "atau", "dan",
    "dengan", "untuk", "dari", "yang", "jika", "bila",
}


def fraksi_ke_float(s: str) -> float:
    """Konversi '1/2' → 0.5, '3/4' → 0.75, '1.5' → 1.5"""
    s = s.replace(",", ".").replace("\\", "/")
    if "/" in s:
        parts = s.split("/")
        try:
            return float(parts[0]) / float(parts[1])
        except:
            return 1.0
    try:
        return float(s)
    except:
        return 1.0


def parse_bahan(baris: str) -> dict:
    """
    Parse satu baris bahan menjadi dict:
    {nama, kuantitas, satuan, gram_estimasi}

    Contoh:
    '3 siung bawang putih (haluskan)'
    → {nama: 'bawang putih', kuantitas: 3.0, satuan: 'siung', gram: 15.0}
    """
    text = str(baris).lower()

    # Hapus tanda kurung
    text = re.sub(r"\(.*?\)", " ", text)
    text = re.sub(r"[^a-z0-9\s/.,]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    kuantitas = 1.0
    satuan    = None
    gram      = DEFAULT_GRAM

    # Cari angka di awal
    m_angka = re.match(r"^" + POLA_ANGKA, text)
    if m_angka:
        kuantitas = fraksi_ke_float(m_angka.group(1))
        if kuantitas <= 0:
            kuantitas = 1.0
        text = text[m_angka.end():].strip()

    # Cari satuan setelah angka
    m_satuan = re.match(r"^(" + SATUAN_REGEX + r")\b", text)
    if m_satuan:
        satuan = m_satuan.group(1)
        gram   = KONVERSI_SATUAN[satuan] * kuantitas
        text   = text[m_satuan.end():].strip()
    else:
        # Tidak ada satuan → pakai default × kuantitas
        gram = DEFAULT_GRAM * kuantitas

    # Bersihkan nama bahan
    words = [w for w in text.split()
             if w not in STOPWORDS_BAHAN
             and not re.match(r"^\d+$", w)
             and len(w) >= 2]
    nama = " ".join(words).strip()

    return {
        "nama":      nama,
        "kuantitas": kuantitas,
        "satuan":    satuan or "default",
        "gram":      round(gram, 2),
    }


# Test parse_bahan
print("Test parse_bahan:")
test_bahan = [
    "3 siung bawang putih (haluskan)",
    "1/2 kg daging sapi has dalam",
    "2 sdm kecap manis",
    "secukupnya garam",
    "1 ekor ayam kampung potong 12",
    "200 gram tempe",
    "1 papan tempe",
    "4 butir telur",
]
for b in test_bahan:
    hasil = parse_bahan(b)
    print(f"  '{b}'")
    print(f"  → nama='{hasil['nama']}' | {hasil['kuantitas']} {hasil['satuan']} | {hasil['gram']}g")
    print()

Test parse_bahan:
  '3 siung bawang putih (haluskan)'
  → nama='bawang putih' | 3.0 siung | 15.0g

  '1/2 kg daging sapi has dalam'
  → nama='daging sapi has dalam' | 0.5 kg | 500.0g

  '2 sdm kecap manis'
  → nama='kecap manis' | 2.0 sdm | 30.0g

  'secukupnya garam'
  → nama='garam' | 1.0 default | 50.0g

  '1 ekor ayam kampung potong 12'
  → nama='ayam kampung' | 1.0 ekor | 800.0g

  '200 gram tempe'
  → nama='tempe' | 200.0 gram | 200.0g

  '1 papan tempe'
  → nama='tempe' | 1.0 papan | 200.0g

  '4 butir telur'
  → nama='telur' | 4.0 butir | 220.0g



In [5]:
# ============================================================
# SIAPKAN LOOKUP NUTRISI
# ============================================================

df_nutrisi["clean_name"] = df_nutrisi["name"].str.lower().str.strip()

# Tangani duplikat dengan rata-rata
df_nut_clean = df_nutrisi.groupby("clean_name", as_index=False)[
    ["calories", "proteins", "fat", "carbohydrate"]
].mean()

nutrisi_lookup = df_nut_clean.set_index("clean_name")[
    ["calories", "proteins", "fat", "carbohydrate"]
].to_dict(orient="index")

nutrisi_names = list(nutrisi_lookup.keys())

print(f"Nutrisi lookup: {len(nutrisi_names)} item unik")

Nutrisi lookup: 1343 item unik


In [6]:
# ============================================================
# FUZZY MATCHING
# ============================================================

FUZZY_THRESHOLD = 70
fuzzy_cache = {}


def cari_nutrisi(nama: str):
    if not nama or len(nama) < 2:
        return None
    if nama in fuzzy_cache:
        return fuzzy_cache[nama]
    if nama in nutrisi_lookup:
        fuzzy_cache[nama] = nutrisi_lookup[nama]
        return nutrisi_lookup[nama]
    hasil = process.extractOne(
        nama, nutrisi_names,
        scorer=fuzz.token_sort_ratio,
        score_cutoff=FUZZY_THRESHOLD
    )
    if hasil:
        r = nutrisi_lookup[hasil[0]]
        fuzzy_cache[nama] = r
        return r
    fuzzy_cache[nama] = None
    return None


print("Fuzzy matching siap.")

Fuzzy matching siap.


In [7]:
# ============================================================
# ESTIMASI JUMLAH PORSI DARI TEKS STEPS
# ============================================================

POLA_PORSI = [
    r"untuk\s+(\d+)\s*(?:orang|porsi|pax|serving)",
    r"(\d+)\s*(?:orang|porsi|pax|serving)",
    r"sajian\s+(\d+)",
    r"(\d+)\s*sajian",
]


def estimasi_porsi(title: str, steps: str) -> float:
    """Estimasi jumlah porsi dari judul atau langkah masak."""
    teks = str(title).lower() + " " + str(steps).lower()
    for pola in POLA_PORSI:
        m = re.search(pola, teks)
        if m:
            n = float(m.group(1))
            if 1 <= n <= 20:   # sanity check
                return n
    return DEFAULT_PORSI   # default 4 porsi


# Test
print("Test estimasi porsi:")
print(f"  'Ayam Goreng untuk 6 orang' → {estimasi_porsi('Ayam Goreng untuk 6 orang', '')} porsi")
print(f"  'Resep untuk 2 orang' → {estimasi_porsi('', 'resep untuk 2 orang')} porsi")
print(f"  'Tanpa info porsi' → {estimasi_porsi('Bakso Sapi', 'rebus daging')} porsi (default)")

Test estimasi porsi:
  'Ayam Goreng untuk 6 orang' → 6.0 porsi
  'Resep untuk 2 orang' → 2.0 porsi
  'Tanpa info porsi' → 4.0 porsi (default)


In [8]:
# ============================================================
# FUNGSI ESTIMASI KALORI PER RESEP (VERSI BARU)
# ============================================================

def estimasi_kalori_resep(row) -> dict:
    """
    Estimasi kalori per PORSI dari satu baris resep.

    Langkah:
    1. Pecah ingredients menjadi list bahan
    2. Parse setiap bahan → kuantitas + satuan + gram
    3. Fuzzy match nama bahan ke nutrition.csv
    4. Hitung kalori = (gram / 100) × kalori_per_100g
    5. Bagi dengan estimasi jumlah porsi
    """
    ingredients_str = row["Ingredients"]
    title           = row.get("Title", "")
    steps           = row.get("Steps", "")

    if pd.isna(ingredients_str) or str(ingredients_str).strip() == "":
        return _empty_result()

    baris_list = str(ingredients_str).split("--")
    porsi      = estimasi_porsi(title, steps)

    total_cal  = total_pro = total_fat = total_carb = 0.0
    matched    = 0
    total_bahan = 0
    matched_names = []

    for baris in baris_list:
        baris = baris.strip()
        if not baris:
            continue

        parsed  = parse_bahan(baris)
        nama    = parsed["nama"]
        gram    = parsed["gram"]

        if not nama or len(nama) < 2:
            continue

        total_bahan += 1
        nutrisi = cari_nutrisi(nama)

        if nutrisi:
            # Kalori dihitung proporsional terhadap gram
            # nutrition.csv menyimpan nilai per 100g
            faktor = gram / 100.0

            total_cal  += nutrisi["calories"]     * faktor
            total_pro  += nutrisi["proteins"]     * faktor
            total_fat  += nutrisi["fat"]          * faktor
            total_carb += nutrisi["carbohydrate"] * faktor
            matched    += 1
            matched_names.append(f"{nama}({gram}g)")

    if matched == 0 or total_bahan == 0:
        return _empty_result()

    # Bagi dengan jumlah porsi
    return {
        "calories_total":     round(total_cal,  1),
        "calories":           round(total_cal  / porsi, 1),  # per porsi
        "proteins":           round(total_pro  / porsi, 2),
        "fat":                round(total_fat  / porsi, 2),
        "carbohydrate":       round(total_carb / porsi, 2),
        "porsi":              porsi,
        "matched_ingredients":matched,
        "total_ingredients":  total_bahan,
        "match_ratio":        round(matched / total_bahan, 3),
        "matched_names":      ", ".join(matched_names),
    }


def _empty_result():
    return {
        "calories_total": np.nan, "calories": np.nan,
        "proteins": np.nan, "fat": np.nan, "carbohydrate": np.nan,
        "porsi": DEFAULT_PORSI, "matched_ingredients": 0,
        "total_ingredients": 0, "match_ratio": 0.0, "matched_names": "",
    }


# ---- Test pada beberapa resep ----
print("Test estimasi kalori (per porsi):")
for i in range(3):
    row = df_resep.iloc[i]
    hasil = estimasi_kalori_resep(row)
    print(f"\n  Resep : {row['Title']}")
    print(f"  Kalori total  : {hasil['calories_total']} kkal")
    print(f"  Porsi         : {hasil['porsi']} orang")
    print(f"  Kalori/porsi  : {hasil['calories']} kkal  ← ini yang disimpan")
    print(f"  Protein/porsi : {hasil['proteins']}g")
    print(f"  Match ratio   : {hasil['match_ratio']}")

Test estimasi kalori (per porsi):

  Resep : Nasi Briyani Simple Ala Mamah Mumtaz
  Kalori total  : 1622.7 kkal
  Porsi         : 4.0 orang
  Kalori/porsi  : 405.7 kkal  ← ini yang disimpan
  Protein/porsi : 24.88g
  Match ratio   : 0.31

  Resep : Bakso Sapi Simple Ga Ribet
  Kalori total  : 1934.3 kkal
  Porsi         : 4.0 orang
  Kalori/porsi  : 483.6 kkal  ← ini yang disimpan
  Protein/porsi : 43.5g
  Match ratio   : 0.583

  Resep : Kroket kentang isi ayam dan wortel
(Indonesian Potato Croquettes with Chicken and carrot)
  Kalori total  : 1911.3 kkal
  Porsi         : 4.0 orang
  Kalori/porsi  : 477.8 kkal  ← ini yang disimpan
  Protein/porsi : 26.59g
  Match ratio   : 0.367


In [9]:
# ============================================================
# PROSES SEMUA RESEP
# ============================================================

print(f"Memproses {len(df_resep):,} resep...")
print("Estimasi waktu: 5-15 menit\n")

hasil_list = []
for _, row in tqdm(df_resep.iterrows(), total=len(df_resep)):
    hasil_list.append(estimasi_kalori_resep(row))

df_estimasi      = pd.DataFrame(hasil_list)
df_resep_nutrisi = pd.concat([df_resep.reset_index(drop=True), df_estimasi], axis=1)

print(f"\nSelesai! Total: {len(df_resep_nutrisi):,} resep")

Memproses 11,514 resep...
Estimasi waktu: 5-15 menit



100%|██████████████████████████████████████████████████████████████████████████████| 11514/11514 [01:28<00:00, 129.71it/s]


Selesai! Total: 11,514 resep


In [10]:
# ============================================================
# STATISTIK & VALIDASI
# ============================================================

total      = len(df_resep_nutrisi)
ada_kalori = df_resep_nutrisi["calories"].notna().sum()

print(f"Total resep         : {total:,}")
print(f"Ada estimasi kalori : {ada_kalori:,} ({ada_kalori/total*100:.1f}%)")

df_valid = df_resep_nutrisi[df_resep_nutrisi["calories"].notna()]

print(f"\nDistribusi kalori PER PORSI:")
print(df_valid["calories"].describe().round(1))

# Cek resep dengan kalori tidak realistis
terlalu_tinggi = (df_valid["calories"] > 1200).sum()
terlalu_rendah = (df_valid["calories"] < 30).sum()
realistis      = ((df_valid["calories"] >= 30) & (df_valid["calories"] <= 1200)).sum()

print(f"\nValidasi kalori per porsi:")
print(f"  Realistis (30-1200 kkal) : {realistis:,} resep")
print(f"  Terlalu tinggi (>1200)   : {terlalu_tinggi:,} resep")
print(f"  Terlalu rendah (<30)     : {terlalu_rendah:,} resep")

Total resep         : 11,514
Ada estimasi kalori : 11,410 (99.1%)

Distribusi kalori PER PORSI:
count     11410.0
mean        591.5
std       10002.3
min           0.0
25%         108.1
50%         236.6
75%         485.8
max      861273.8
Name: calories, dtype: float64

Validasi kalori per porsi:
  Realistis (30-1200 kkal) : 9,969 resep
  Terlalu tinggi (>1200)   : 537 resep
  Terlalu rendah (<30)     : 904 resep


In [11]:
# ============================================================
# FILTER FINAL
# ============================================================

df_final = df_resep_nutrisi[
    df_resep_nutrisi["calories"].notna() &
    (df_resep_nutrisi["calories"] >= 30)   &   # minimal 30 kkal/porsi
    (df_resep_nutrisi["calories"] <= 1200) &   # maksimal 1200 kkal/porsi
    (df_resep_nutrisi["match_ratio"] >= 0.2)   # minimal 20% bahan teridentifikasi
].copy().reset_index(drop=True)

print(f"Dataset final setelah filter: {len(df_final):,} resep")
print(f"\nDistribusi per kategori:")
print(df_final["Kategori"].value_counts())

print(f"\nDistribusi kalori (realistis):")
bins   = [0, 200, 300, 400, 500, 700, 1200]
labels = ["<200", "200-300", "300-400", "400-500", "500-700", "700-1200"]
df_final["kalori_group"] = pd.cut(df_final["calories"], bins=bins, labels=labels)
print(df_final["kalori_group"].value_counts().sort_index())

Dataset final setelah filter: 9,873 resep

Distribusi per kategori:
Kategori
ikan.csv       1369
ayam.csv       1259
tempe.csv      1258
telur.csv      1255
sapi.csv       1243
tahu.csv       1185
kambing.csv    1157
udang.csv      1147
Name: count, dtype: int64

Distribusi kalori (realistis):
kalori_group
<200        4016
200-300     1664
300-400     1115
400-500      863
500-700     1084
700-1200    1131
Name: count, dtype: int64


In [12]:
# ============================================================
# SIMPAN
# ============================================================

final_cols = [
    "Title", "Ingredients", "Steps", "Loves", "Kategori",
    "calories", "proteins", "fat", "carbohydrate",
    "calories_total", "porsi",
    "matched_ingredients", "total_ingredients",
    "match_ratio", "matched_names",
]

df_output = df_final[final_cols].copy()
df_output.to_csv("../data/dataset_final_resep_nutrisi.csv", index=False)

print("============================================")
print("SELESAI")
print("============================================")
print(f"Saved : data/dataset_final_resep_nutrisi.csv")
print(f"Total : {len(df_output):,} resep")
print()
print("Preview (kalori sudah per porsi):")
df_output[["Title", "Kategori", "calories", "porsi", "match_ratio"]].head(10)

SELESAI
Saved : data/dataset_final_resep_nutrisi.csv
Total : 9,873 resep

Preview (kalori sudah per porsi):


,Title,Kategori,calories,porsi,match_ratio
0,Nasi Briyani Simple Ala Mamah Mumtaz,kambing.csv,405.7,4.0,0.310
1,Bakso Sapi Simple Ga Ribet,sapi.csv,483.6,4.0,0.583
2,Kroket kentang isi ayam dan wortel\n(Indonesia...,ayam.csv,477.8,4.0,0.367
3,Sate Telur Gulung Crispy Ekonomis Isi Sosis | ...,telur.csv,125.2,4.0,0.364
4,Siomay Ayam ekonomis,ayam.csv,622.0,4.0,0.400
5,Martabak Gule Daging,kambing.csv,660.0,4.0,0.238
6,Nasi Kebuli Kambing Nikmat !,kambing.csv,413.8,4.0,0.353
7,"Siomay Ayam Udang,,, Enakkk + tips",udang.csv,851.0,4.0,0.565
8,Qabli Rice - Nasi Kabuli,kambing.csv,266.3,2.0,0.316
9,Tahu Bulat Isi Daging | Lamb-Stuffed Round Tofu,kambing.csv,105.1,4.0,0.286
